# 03 — Usar un LLM como componente de una aplicación

## Tercer notebook del bloque LLM

En el notebook anterior vimos el mecanismo interno básico:

```text
tokens → modelo → logits → siguiente token → generación
```

Ahora cambiaremos de nivel de abstracción. En una aplicación real normalmente no inspeccionaremos logits. Utilizaremos un LLM mediante una API:

```text
nuestra aplicación
      │
      ├── instrucciones
      ├── pregunta del usuario
      └── contexto
      │
      ▼
     LLM
      │
      ▼
   respuesta
```

El objetivo de este notebook es aprender a tratar el LLM como **un componente software**, antes de conectarlo con la búsqueda semántica del notebook 01.

Todavía **no construiremos RAG**. El contexto será manual para poder aislar el comportamiento del LLM.

## 0. Dependencia y API key

Instala el SDK oficial:

```bash
uv add openai python-dotenv
```

La API key **no debe escribirse en el notebook ni subirse a Git**.

Crea un archivo `.env` en la raíz del repositorio (ya está en `.gitignore`):

```text
OPENAI_API_KEY=sk-...
```

El notebook carga ese archivo con `load_dotenv()`. Si `OPENAI_API_KEY` ya existe como variable de entorno, se respeta ese valor.

También puedes elegir el modelo añadiendo otra línea al `.env`:

```text
OPENAI_MODEL=gpt-6-luna
```

Si no defines `OPENAI_MODEL`, este notebook utilizará `gpt-6-luna` como modelo por defecto.

## 1. Importaciones y configuración

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "Falta OPENAI_API_KEY. Añádela al archivo .env en la raíz del repositorio."
    )

client = OpenAI()
print("Model:", MODEL)

## 2. Primera llamada al LLM

Utilizaremos la **Responses API**.

La llamada mínima tiene dos piezas importantes:

- `model`: qué modelo queremos utilizar;
- `input`: qué queremos preguntarle.

In [ ]:
response = client.responses.create(
    model=MODEL,
    input="Explica en dos frases qué es la gestión del tráfico aéreo."
)

print(response.output_text)

Antes implementábamos nosotros tokenización, forward pass, logits y decoding. Ahora la API encapsula ese proceso en `client.responses.create(...)`, aunque conceptualmente el modelo sigue generando tokens.

# 3. Separar instrucciones y petición del usuario

In [ ]:
instructions = """
Eres un asistente técnico especializado en sistemas ATM.
Responde en español.
Sé conciso.
No inventes datos operacionales que no aparezcan en el contexto proporcionado.
Si falta información para responder, indícalo explícitamente.
"""

user_question = "¿Por qué puede reducirse la capacidad de un sector?"

response = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input=user_question,
)

print(response.output_text)

Conceptualmente:

```text
INSTRUCTIONS ── cómo debe comportarse
USER INPUT   ── qué quiere el usuario
       │
       ▼
      LLM
```

# 4. El problema del conocimiento no proporcionado

In [ ]:
question = """
¿Cuál es la causa exacta de la reducción de capacidad
del sector Madrid-Norte a las 14:30 de hoy?
"""

response = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input=question,
)

print(response.output_text)

Aunque un modelo pueda poseer conocimiento general sobre ATM, no debemos asumir que conoce el estado actual de nuestros sistemas, documentos internos, incidencias privadas o información operacional en tiempo real.

# 5. Proporcionar contexto manualmente

In [ ]:
context = """
[ATM-INC-104]
Hora: 14:22
Sector: Madrid-Norte
La demanda prevista supera temporalmente la capacidad declarada del sector.

[ATM-INC-107]
Hora: 14:25
Sector: Madrid-Norte
Se ha reducido la configuración disponible debido a una ausencia imprevista de personal.

[ATM-WX-031]
Hora: 14:27
Sector: Madrid-Norte
No existen fenómenos meteorológicos significativos que afecten al sector.
"""

print(context)

El contexto **no modifica los parámetros del modelo**. Simplemente forma parte de la entrada que recibe el LLM en esta llamada.

In [ ]:
question = "¿Cuál es la causa de la reducción de capacidad del sector Madrid-Norte?"

input_with_context = f"""
Usa exclusivamente el siguiente CONTEXTO para responder.

CONTEXTO:
{context}

PREGUNTA:
{question}
"""

response = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input=input_with_context,
)

print(response.output_text)

Esto ya tiene la forma:

```text
datos de nuestra aplicación
        ↓
      contexto
        +
   pregunta usuario
        +
    instrucciones
        ↓
       LLM
        ↓
respuesta basada en contexto
```

Todavía no es RAG porque **hemos seleccionado el contexto a mano**.

# 6. Preguntas que el contexto NO puede responder

In [ ]:
question = "¿Cuántos controladores estaban trabajando en el sector Madrid-Norte?"

input_with_context = f"""
Usa exclusivamente el siguiente CONTEXTO para responder.
Si el contexto no contiene la respuesta, di claramente que no se puede determinar.

CONTEXTO:
{context}

PREGUNTA:
{question}
"""

response = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input=input_with_context,
)

print(response.output_text)

Un buen comportamiento también consiste en reconocer cuándo la evidencia disponible no es suficiente. En RAG, recuperar documentos no implica que esos documentos contengan necesariamente la respuesta.

# 7. El prompt es parte del diseño de la aplicación

In [ ]:
question = "Resume la situación del sector Madrid-Norte."

for style in [
    "Responde en una sola frase.",
    "Responde con tres puntos: causa, factores descartados y conclusión.",
]:
    response = client.responses.create(
        model=MODEL,
        instructions=f"""
        Eres un asistente técnico ATM.
        Usa exclusivamente la información proporcionada.
        {style}
        """,
        input=f"""
        CONTEXTO:
        {context}

        PREGUNTA:
        {question}
        """,
    )

    print("=" * 90)
    print(style)
    print(response.output_text)
    print()

El modelo y el contexto son los mismos; cambia la especificación de la tarea. Por eso el prompt forma parte del comportamiento del sistema.

# 8. Encapsular la llamada en una función

In [ ]:
ATM_INSTRUCTIONS = """
Eres un asistente técnico especializado en ATM.

Reglas:
1. Responde en español.
2. Basa la respuesta únicamente en el contexto suministrado.
3. No inventes hechos ausentes del contexto.
4. Si no existe evidencia suficiente, indícalo.
5. Cuando sea posible, menciona los identificadores de las fuentes utilizadas.
"""

def ask_atm_llm(question, context):
    prompt = f"""
CONTEXTO:
{context}

PREGUNTA:
{question}
"""

    response = client.responses.create(
        model=MODEL,
        instructions=ATM_INSTRUCTIONS,
        input=prompt,
    )

    return response.output_text

In [ ]:
answer = ask_atm_llm(
    question="¿Qué factores explican la reducción de capacidad?",
    context=context,
)

print(answer)

Ahora nuestra aplicación tiene una interfaz sencilla:

```python
ask_atm_llm(question, context)
```

# 9. La conexión con el notebook 01

En `01_embeddings_semantic_search.ipynb` construimos conceptualmente `retrieve(question)`:

```text
pregunta → embedding → semantic search → TOP-K documentos
```

En este notebook hemos construido `ask_atm_llm(question, context)`:

```text
pregunta + contexto → LLM → respuesta
```

El siguiente paso será unir ambas piezas:

```python
documents = retrieve(question)
context = build_context(documents)
answer = ask_atm_llm(question, context)
```

Eso será nuestro primer **RAG**.

# 10. RAG no significa entrenar el LLM

Cuando introducimos documentos en el contexto **no estamos entrenando ni haciendo fine-tuning**.

```text
PRETRAINING / FINE-TUNING
→ cambia parámetros

PROMPT / CONTEXT / RAG
→ no cambia parámetros
→ modifica la información disponible en esa ejecución
```

# 11. ¿Por qué no poner todos los documentos en el prompt?

Una organización puede tener miles de documentos, manuales, requisitos, incidencias y especificaciones. Enviar todo en cada consulta sería innecesario, más caro, más lento y limitado por la ventana de contexto.

Por eso necesitamos retrieval:

```text
gran corpus
    ↓
buscar solo lo relevante
    ↓
TOP-K fragmentos
    ↓
LLM
```

# 12. Aplicación al proyecto que queremos construir

Nuestro objetivo final puede tener esta forma:

```text
DOCUMENTO DE PRUEBA
      ↓
LLM
      ↓
interpretación de pasos
      ↓
selección de herramientas MCP
      ↓
ejecución del sistema
      ↓
resultados
```

Ahí convivirán varias capacidades:

- **RAG** para recuperar documentación, procedimientos, ejemplos y restricciones;
- **Tool calling / MCP** para consultar y actuar sobre el sistema;
- **LLM** para interpretar intención, decidir qué necesita y combinar resultados.

# 13. Experimentos

Prueba a modificar:

1. las `instructions`;
2. el contexto;
3. una pregunta cuya respuesta sí está en el contexto;
4. otra cuya respuesta no está;
5. información contradictoria en dos fuentes;
6. una instrucción del usuario que intente hacer que el modelo ignore las reglas.

No buscamos todavía resolver todos esos problemas; queremos detectar que existen.

# 14. Preguntas de comprobación

Al terminar deberías poder explicar:

1. ¿Qué diferencia existe entre `instructions` e `input`?
2. ¿Dónde está la información privada de nuestra aplicación?
3. ¿Añadir contexto modifica los parámetros del LLM?
4. ¿Por qué un LLM puede conocer ATM general y no una incidencia interna?
5. ¿Por qué debemos indicar qué hacer si el contexto es insuficiente?
6. ¿Qué diferencia existe entre prompting, RAG y fine-tuning?
7. ¿Por qué no enviar siempre todos nuestros documentos al LLM?
8. ¿Qué función cumple retrieval?
9. ¿Qué construimos en el notebook 01?
10. ¿Qué construimos en este notebook?
11. ¿Qué obtenemos al conectar ambos?
12. ¿Dónde encajarán posteriormente las herramientas MCP?

# 15. Siguiente paso — nuestro primer RAG ATM

Ya tenemos:

```text
01_embeddings_semantic_search → retrieve()
02_llm_basics               → entendemos la generación
03_llm_application_basics   → ask_atm_llm(question, context)
```

Ahora construiremos:

```text
pregunta → embedding → retrieval → TOP-K → contexto → LLM → respuesta
```

Ese será **`04_rag_atm.ipynb`**.